<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%203/2_st445_week3_lab2_pandas_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 3: getting familiar with pandas

**Lecture companion notebook · run it alongside the pandas part of the lecture, or on your own afterwards**

This notebook introduces the pandas operations you will use every week: loading data, looking at it, selecting rows and columns, filtering, arithmetic, functions and `lambda`, missing values, sorting and grouping. Every section explains what the code does and what the output should look like, and most end with a short **Try it** exercise.

### The dataset

**Adult (census income)**, from the **UCI Machine Learning Repository**: 48,842 people from the 1994 US Census, with age, education, occupation, hours worked per week, capital gains and losses, and whether their income was above \$50,000 a year. It is small, it is about economics, and it has the problems real data has: text categories, hyphenated column names, missing values, and inconsistent labels.

> Becker, B. and Kohavi, R. (1996). *Adult* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20. Licensed under CC BY 4.0.

Cell marks: 🟢 run and read · 🔵 **Try it**: write a few lines yourself, then run the check. **Method cards** explain each new function and the minimum you need to set.

**Before you start:** **File → Save a copy in Drive**.

## 0. Set-up

In [1]:
# 🟢 Run this first
import numpy as np
import pandas as pd

def check(condition, hint="Not yet. Read the task again."):
    print("✅ Correct" if condition else f"❌ {hint}")

pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__)

pandas 2.2.3


---
## 1. Loading a dataset from the UCI repository

The UCI repository has a small Python client, **`ucimlrepo`**, which calls the repository's API and returns the data as pandas DataFrames, together with its metadata. Each dataset has an id; Adult is id **2**.

> **Method card: `fetch_ucirepo(id=...)`** returns an object with:
> * `.data.features`: a DataFrame of the explanatory variables;
> * `.data.targets`: a DataFrame of the variable to predict (here, income);
> * `.metadata`: a dictionary describing the dataset (title, number of rows, licence, citation...);
> * `.variables`: a table describing every column (name, type, whether it has missing values).

In [2]:
# 🟢 Install the client if Colab does not have it, then fetch the data through the API
try:
    from ucimlrepo import fetch_ucirepo
except ImportError:
    %pip install --quiet ucimlrepo
    from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
print(adult.metadata["name"], "|", adult.metadata["num_instances"], "rows")
adult.variables[["name", "role", "type", "missing_values"]]

Adult | 48842 rows


,name,role,type,missing_values
0,age,Feature,Integer,no
1,workclass,Feature,Categorical,yes
2,fnlwgt,Feature,Integer,no
3,education,Feature,Categorical,no
4,education-num,Feature,Integer,no
5,marital-status,Feature,Categorical,no
6,occupation,Feature,Categorical,yes
7,relationship,Feature,Categorical,no
8,race,Feature,Categorical,no
9,sex,Feature,Binary,no


**What you should see:** the dataset's name and size, then a table describing the 15 variables: their role (`Feature` or `Target`), their type (`Integer` or `Categorical`) and whether they have missing values. Reading the documentation of a dataset **before** the data itself is the same habit as reading an API's documentation in Week 2.

Now combine the features and the target into one DataFrame.

> **Method card: `pd.concat([df1, df2], axis=1)`** puts DataFrames side by side, matching rows by their index. (`axis=0`, the default, stacks them on top of each other.)

In [3]:
# 🟢 One table: features and target side by side
df = pd.concat([adult.data.features, adult.data.targets], axis=1)
print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


**What you should see:** `(48842, 15)`: 48,842 rows and 15 columns. `head()` shows the first five rows.

### The same data as a plain CSV file

The repository also serves the original files. Reading a CSV from a URL is one line, but this file has **no header row**, values preceded by a space, and missing values written as `?`. Each problem has a `read_csv` parameter.

> **Method card: `pd.read_csv(path_or_url, ...)`** reads a comma-separated file. The parameters used here:
> * `header=None`: the file has no header row;
> * `names=[...]`: the column names to use instead;
> * `na_values="?"`: treat `?` as a missing value;
> * `skipinitialspace=True`: ignore the space after each comma.
>
> Without `na_values`, the `?` would stay as text and `isna()` would find nothing (section 8).

In [4]:
# 🟢 Optional: read the original training file directly (works if the UCI site is reachable)
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
NAMES = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", "occupation",
         "relationship", "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]
try:
    csv_df = pd.read_csv(URL, header=None, names=NAMES, na_values="?", skipinitialspace=True)
    print(csv_df.shape)       # the training file only: 32,561 rows
except Exception as err:
    print("Could not read the CSV:", type(err).__name__, "- the API version above is enough for this notebook.")

(32561, 15)


**What you should see:** `(32561, 15)`: the original **training** file. The API version has 48,842 rows because it combines the training and test files. Same source, different files: one reason to record exactly what you downloaded (Week 2, provenance).

### Tidy the column names

Names such as `education-num` contain hyphens, so they cannot be used with dot notation (`df.education-num` would mean "`df.education` minus `num`"). Replace hyphens with underscores once, at the start.

In [5]:
# 🟢 Rename columns: hyphens to underscores
df.columns = df.columns.str.replace("-", "_")
print(df.columns.tolist())

['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


---
## 2. A first look

Five methods tell you most of what you need about a new table.

| Method | Tells you |
|---|---|
| `df.shape` | the number of rows and columns |
| `df.head(n)`, `df.tail(n)` | the first or last `n` rows |
| `df.dtypes`, `df.info()` | each column's type, and how many values are not missing |
| `df.describe()` | summary statistics for the numeric columns |
| `df["col"].value_counts()` | how often each value appears in one column |

In [6]:
# 🟢 Types and missing counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education_num   48842 non-null  int64 
 5   marital_status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital_gain    48842 non-null  int64 
 11  capital_loss    48842 non-null  int64 
 12  hours_per_week  48842 non-null  int64 
 13  native_country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


**What you should see:** 15 columns. Six are integers (`int64`): `age`, `fnlwgt`, `education_num`, `capital_gain`, `capital_loss`, `hours_per_week`. The rest are text (`str` in pandas 3.0, `object` in 2.x). The *Non-Null Count* is lower than 48,842 for `workclass`, `occupation` and `native_country`: those columns have missing values.

In [7]:
# 🟢 Summary statistics of the numeric columns
df.describe().round(1)

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,48842.0,48842.0,48842.0,48842.0,48842.0,48842.0
mean,38.6,189664.1,10.1,1079.1,87.5,40.4
std,13.7,105604.0,2.6,7452.0,403.0,12.4
min,17.0,12285.0,1.0,0.0,0.0,1.0
25%,28.0,117550.5,9.0,0.0,0.0,40.0
50%,37.0,178144.5,10.0,0.0,0.0,40.0
75%,48.0,237642.0,12.0,0.0,0.0,45.0
max,90.0,1490400.0,16.0,99999.0,4356.0,99.0


**What you should see:** for each numeric column, the count, mean, standard deviation, minimum, quartiles and maximum. Things worth noticing: the median age is 37; `hours_per_week` runs from 1 to 99; `capital_gain` has a median of 0 but a maximum of 99,999, a very skewed column (and 99,999 looks like a cap or code: worth checking in Week 4).

In [8]:
# 🟢 Text columns: how many distinct values, and the most common
print(df.describe(exclude="number").T)          # count, distinct values, most common value and its frequency
print()
print(df["income"].value_counts())

                count unique                 top   freq
workclass       47879      9             Private  33906
education       48842     16             HS-grad  15784
marital_status  48842      7  Married-civ-spouse  22379
occupation      47876     15      Prof-specialty   6172
relationship    48842      6             Husband  19716
race            48842      5               White  41762
sex             48842      2                Male  32650
native_country  48568     42       United-States  43832
income          48842      4               <=50K  24720

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64


**What you should see:** `income` has **four** distinct values, not two: `<=50K`, `>50K`, `<=50K.` and `>50K.`. The test file wrote its labels with a full stop. Inconsistent labels are a cleaning problem (Week 4); we fix this one now because every later section uses income.

In [9]:
# 🟢 Make the income labels consistent: remove a trailing full stop
df["income"] = df["income"].str.rstrip(".")
df["income"].value_counts()

,count
income,
<=50K,37155
>50K,11687


---
## 3. Selecting columns

A **column** of a DataFrame is a **Series**: a one-dimensional labelled array. Selecting one column gives a Series; selecting a **list** of columns gives a DataFrame.

In [10]:
# 🟢 One column: a Series
ages = df["age"]
print(type(ages).__name__, ages.shape)

# 🟢 Several columns: a DataFrame (note the double brackets: a list inside the brackets)
people = df[["age", "education", "hours_per_week"]]
print(type(people).__name__, people.shape)

# 🟢 Dot notation works for simple names, but brackets always work
print(df.age.equals(df["age"]))

Series (48842,)
DataFrame (48842, 3)
True


**What you should see:** `Series (48842,)`, then `DataFrame (48842, 3)`, then `True`. Prefer brackets: they work for every column name, including names with spaces or names that clash with methods (a column called `count` would be hidden by `df.count`).

> 🔵 **Try it.** Make a DataFrame `work` with only the columns `occupation`, `hours_per_week` and `income`, in that order.

In [11]:
work = df[["occupation", "hours_per_week", "income"]]

In [12]:
check(list(work.columns) == ["occupation", "hours_per_week", "income"] and len(work) == len(df), "Pass a list of the three names inside the brackets.")

✅ Correct


---
## 4. The index, and selecting rows with `loc` and `iloc`

Every DataFrame has an **index**: the labels of its rows. Here it is the default, a `RangeIndex` from 0, so labels and positions happen to coincide. That hides the difference between selecting by **label** and by **position**, so first make a small table with a text index.

> **Method card: row selection**
>
> | Method | Selects by | Slices |
> |---|---|---|
> | `df.loc[rows, columns]` | **label** | include the end: `df.loc["a":"c"]` gives a, b, c |
> | `df.iloc[rows, columns]` | integer **position** | exclude the end, as in Python: `df.iloc[0:2]` gives 2 rows |
> | `df.at[row, column]`, `df.iat[i, j]` | one element, by label or by position | |

In [13]:
# 🟢 A small table with a text index
small = df.loc[:4, ["age", "education", "occupation", "hours_per_week"]].copy()
small.index = ["p1", "p2", "p3", "p4", "p5"]
small

,age,education,occupation,hours_per_week
p1,39,Bachelors,Adm-clerical,40
p2,50,Bachelors,Exec-managerial,13
p3,38,HS-grad,Handlers-cleaners,40
p4,53,11th,Handlers-cleaners,40
p5,28,Bachelors,Prof-specialty,40


In [14]:
# 🟢 By label with loc
print(small.loc["p2"], "\n")                              # one row: a Series
print(small.loc["p2":"p4", ["age", "education"]], "\n")   # rows p2 to p4 INCLUDED, two columns

# 🟢 By position with iloc
print(small.iloc[0:2], "\n")                              # positions 0 and 1 only: p1 and p2
print(small.iloc[-1, 0], small.at["p5", "age"], small.iat[4, 0])   # three ways to reach one element

age                            50
education               Bachelors
occupation        Exec-managerial
hours_per_week                 13
Name: p2, dtype: object 

    age  education
p2   50  Bachelors
p3   38    HS-grad
p4   53       11th 

    age  education       occupation  hours_per_week
p1   39  Bachelors     Adm-clerical              40
p2   50  Bachelors  Exec-managerial              13 

28 28 28


**What you should see:** `loc["p2":"p4"]` returns **three** rows, because label slices include their end; `iloc[0:2]` returns **two**, because position slices exclude it. The last line prints the same age three times: by position from the end (`iloc[-1, 0]`), by labels (`at`) and by positions (`iat`).

### Setting an index for a reason

An index is useful when it is a meaningful **key**. Adult has no person identifier, so a meaningful index here is the default one. In Week 2 data, `(iso3, year)` was the natural key. The operation is `set_index`, and the check is `index.is_unique`:

In [15]:
# 🟢 Is education_num a good key? No: many people share it
print("unique:", df.set_index("education_num").index.is_unique)

# 🟢 A row label lookup on the default index, and the label versus position distinction on a sorted table
by_age = df.sort_values("age")
print(by_age.loc[0, "age"], "<- label 0 is still the first person in the file")
print(by_age.iloc[0]["age"], "<- position 0 is now the youngest person")

unique: False
39 <- label 0 is still the first person in the file
17 <- position 0 is now the youngest person


**What you should see:** `unique: False`, so `education_num` would be a bad key. After sorting, **label** 0 still refers to the same person as before, but **position** 0 is now the youngest: sorting moves rows, and labels travel with them (the lecture's alignment slides).

> 🔵 **Try it.** From `df`, select rows with **labels** 10 to 14 and the columns `age` to `education` (all columns between them), storing the result in `block`. Hint: `loc` slices work on columns too.

In [16]:
block = df.loc[10:14, "age":"education"]

In [17]:
check(block.shape == (5, 4) and list(block.columns) == ["age", "workclass", "fnlwgt", "education"],
      "Use df.loc[10:14, 'age':'education']: both label slices include their end.")

✅ Correct


---
## 5. Selecting rows with conditions

A condition on a column gives a **boolean Series**: True or False for each row. Putting it inside the brackets keeps the rows where it is True.

> **Method card: combining conditions**
> * `&` (and), `|` (or), `~` (not). **Each condition needs its own brackets**: `(df.age > 30) & (df.sex == "Female")`. Python's `and` and `or` do not work on Series.
> * `.isin([...])`: is the value in a list?
> * `.between(a, b)`: between `a` and `b`, **inclusive** by default.
> * `.str.contains("text")`: does the text contain this substring?
> * `df.query("age > 30 and sex == 'Female'")`: the same filter as a readable string.

In [18]:
# 🟢 One condition
over_60 = df[df["age"] > 60]
print(len(over_60), "people over 60")

# 🟢 Two conditions: brackets around each, & between them
women_high = df[(df["sex"] == "Female") & (df["income"] == ">50K")]
print(len(women_high), "women with income above 50K")

# 🟢 isin, between, str.contains, and ~ for "not"
degree = df[df["education"].isin(["Bachelors", "Masters", "Doctorate"])]
full_time = df[df["hours_per_week"].between(35, 45)]
managers = df[df["occupation"].str.contains("managerial", case=False, na=False)]
not_private = df[~(df["workclass"] == "Private")]
print(len(degree), len(full_time), len(managers), len(not_private))

3606 people over 60
1769 women with income above 50K
11276 29746 6086 14936


**What you should see:** four or five counts. Note `na=False` in `str.contains`: occupation has missing values, and without it the result would contain missing values instead of False, which cannot be used to filter.

The same filter with `query`, and with `loc` to select columns at the same time:

In [19]:
# 🟢 query, and loc with a condition and a list of columns
q = df.query("sex == 'Female' and income == '>50K'")
print(len(q) == len(women_high))

df.loc[df["hours_per_week"] > 80, ["age", "occupation", "hours_per_week"]].head()

True


,age,occupation,hours_per_week
272,50,Farming-fishing,98
935,37,Craft-repair,99
1172,25,Farming-fishing,99
1417,31,Sales,90
1730,24,Exec-managerial,90


> 🔵 **Try it.** Count the people aged 25 to 35 (inclusive) who work **more than** 50 hours a week and earn above 50K. Store the number in `n_long_hours`.

In [20]:
n_long_hours = len(df[df["age"].between(25, 35) & (df["hours_per_week"] > 50) & (df["income"] == ">50K")])
print(n_long_hours)

487


In [21]:
expected = int(((df.age >= 25) & (df.age <= 35) & (df.hours_per_week > 50) & (df.income == ">50K")).sum())
check(n_long_hours == expected, "Combine three conditions with &, each in its own brackets.")

✅ Correct


---
## 6. Arithmetic on columns

Arithmetic on Series is **vectorised**: it applies to every row at once, like NumPy ufuncs. A Series and a number broadcast; two Series are matched **by index label**.

In [22]:
# 🟢 New columns from arithmetic
df["capital_net"] = df["capital_gain"] - df["capital_loss"]          # Series minus Series
df["hours_per_year"] = df["hours_per_week"] * 52                       # Series times a number
df["age_in_2026"] = df["age"] + (2026 - 1994)                          # the census year was 1994
df[["capital_gain", "capital_loss", "capital_net", "hours_per_week", "hours_per_year", "age", "age_in_2026"]].head()

,capital_gain,capital_loss,capital_net,hours_per_week,hours_per_year,age,age_in_2026
0,2174,0,2174,40,2080,39,71
1,0,0,0,13,676,50,82
2,0,0,0,40,2080,38,70
3,0,0,0,40,2080,53,85
4,0,0,0,40,2080,28,60


**What you should see:** three new columns, computed for all 48,842 rows in one line each. No loop.

### Aggregations, and comparing with the whole

Reductions such as `mean`, `median`, `sum`, `min`, `max` and `std` return one number per column. Combining a column with its own mean is broadcasting (Part 1 of the lecture).

In [23]:
# 🟢 Aggregations
print("mean hours:", round(df["hours_per_week"].mean(), 1), "| median:", df["hours_per_week"].median())
print("share earning above 50K:", round((df["income"] == ">50K").mean(), 3))   # mean of True/False = a share

# 🟢 Each person's hours relative to the average: a Series minus a number
df["hours_vs_mean"] = df["hours_per_week"] - df["hours_per_week"].mean()

# 🟢 A vectorised if: np.where
df["full_time"] = np.where(df["hours_per_week"] >= 35, "full time", "part time")
df["full_time"].value_counts()

mean hours: 40.4 | median: 40.0
share earning above 50K: 0.239


,count
full_time,
full time,40447
part time,8395


**What you should see:** a mean of about 40 hours and a median of exactly 40; about 24% of people earn above 50K (the mean of a True/False Series is the **share** that are True); and the counts of full-time and part-time workers.

### Alignment in arithmetic

Arithmetic between two Series matches **labels**, not positions. If the labels do not match, the result is NaN.

In [24]:
# 🟢 Two Series with different index labels
a = df["hours_per_week"].iloc[:3]          # labels 0, 1, 2
b = df["hours_per_week"].iloc[1:4]         # labels 1, 2, 3
print(a + b)

0     NaN
1    26.0
2    80.0
3     NaN
Name: hours_per_week, dtype: float64


**What you should see:** four rows, labels 0 to 3, with NaN at 0 and 3: those labels appear in only one of the two Series. This is the lecture's opening question again, on real data.

> 🔵 **Try it.** Create a column `gain_per_hour`: capital gain divided by hours per **year**. Then store in `top_gain` the largest value of that column.

In [25]:
df["gain_per_hour"] = df["capital_gain"] / df["hours_per_year"]
top_gain = df["gain_per_hour"].max()
print(round(top_gain, 2))

218.56


In [26]:
check(np.isclose(top_gain, (df.capital_gain / (df.hours_per_week * 52)).max()), "Divide capital_gain by hours_per_year, then take .max().")

✅ Correct


---
## 7. Functions, `lambda`, `map` and `apply`

Sometimes a transformation is easier to write as a function. A **`lambda`** is a function written in one line, without a name: `lambda x: x * 2` takes `x` and returns `x * 2`.

> **Method card: applying functions**
>
> | Method | Applies | Use for |
> |---|---|---|
> | `series.map(dict)` | a lookup, value by value | recoding labels |
> | `series.apply(func)` | a function to each value | logic that has no vectorised form |
> | `df.apply(func, axis=1)` | a function to each **row** | combining several columns (slow on large tables) |
> | `df.assign(new=lambda d: ...)` | creates a column from the DataFrame `d` | building columns in a readable chain |
>
> Prefer vectorised arithmetic or `np.where` when they exist: `apply` runs a Python loop underneath.

In [27]:
# 🟢 A lambda on one column: age bands
df["age_band"] = df["age"].apply(lambda x: f"{(x // 10) * 10}s")      # 37 -> "30s"
print(df[["age", "age_band"]].head(3), "\n")

# 🟢 map with a dictionary: recode labels
df["high_income"] = df["income"].map({">50K": 1, "<=50K": 0})
print(df["high_income"].value_counts(), "\n")

# 🟢 apply across a row: several columns at once (fine here, slow on very large tables)
df["long_hours_manager"] = df.apply(
    lambda row: row["hours_per_week"] > 50 and row["occupation"] == "Exec-managerial", axis=1)
print(df["long_hours_manager"].sum(), "managers working over 50 hours")

   age age_band
0   39      30s
1   50      50s
2   38      30s 

high_income
0    37155
1    11687
Name: count, dtype: int64 

1134 managers working over 50 hours


**What you should see:** an `age_band` column (`30s`, `50s`...), a 0/1 `high_income` column with the same counts as `income`, and a count of managers working long hours.

### The same results, vectorised

Each of the three can be written without `apply`, and the vectorised form is faster.

In [28]:
# 🟢 Vectorised equivalents
band_v = (df["age"] // 10 * 10).astype(str) + "s"
manager_v = (df["hours_per_week"] > 50) & (df["occupation"] == "Exec-managerial")
print(band_v.equals(df["age_band"]), manager_v.equals(df["long_hours_manager"]))

%timeit df.apply(lambda row: row["hours_per_week"] > 50 and row["occupation"] == "Exec-managerial", axis=1)
%timeit (df["hours_per_week"] > 50) & (df["occupation"] == "Exec-managerial")

True True
The slowest run took 4.97 times longer than the fastest. This could mean that an intermediate result is being cached.
755 ms ± 344 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
4.02 ms ± 83 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**What you should see:** `True True`, then two timings: `apply` with `axis=1` takes hundreds of milliseconds; the vectorised version takes about a millisecond. Same answer, hundreds of times faster.

### `assign` with `lambda`, for readable chains

`assign` returns a **new** DataFrame with extra columns; inside it, `lambda d:` refers to the DataFrame as it is at that point of the chain.

In [29]:
# 🟢 A chain: filter, add two columns, sort, and show the top rows
(df[df["capital_gain"] > 0]
   .assign(gain_k=lambda d: d["capital_gain"] / 1000,
           gain_share=lambda d: d["capital_gain"] / d["capital_gain"].sum())
   .sort_values("gain_k", ascending=False)
   [["age", "occupation", "gain_k", "gain_share"]]
   .head())

,age,occupation,gain_k,gain_share
13499,50,Sales,99.999,0.001897
12655,36,Exec-managerial,99.999,0.001897
12677,47,Exec-managerial,99.999,0.001897
47739,32,Exec-managerial,99.999,0.001897
13997,48,Exec-managerial,99.999,0.001897


> 🔵 **Try it.** Using `map` with a dictionary, create `sex_code` with 1 for `"Female"` and 0 for `"Male"`. Then, using `apply` with a `lambda`, create `edu_level`: `"degree"` if `education_num` is 13 or more, otherwise `"no degree"`.

In [30]:
df["sex_code"] = df["sex"].map({"Female": 1, "Male": 0})
df["edu_level"] = df["education_num"].apply(lambda n: "degree" if n >= 13 else "no degree")

In [31]:
check(set(df["sex_code"].dropna().unique()) == {0, 1} and df["sex_code"].sum() == (df.sex == "Female").sum(), "map({'Female': 1, 'Male': 0}).")
check((df["edu_level"] == "degree").sum() == (df.education_num >= 13).sum(), "lambda n: 'degree' if n >= 13 else 'no degree'.")

✅ Correct
✅ Correct


---
## 8. Identifying missing values

pandas marks missing values as `NaN` (or `pd.NA`, or `NaT` for dates). They are not equal to anything, including themselves, so test for them with `isna()`, never with `== np.nan`.

> **Method card: missing values**
> * `df.isna()`: a True/False table, True where a value is missing; `notna()` is the opposite.
> * `df.isna().sum()`: the number of missing values in each column; `.mean()` gives the share.
> * `df[df["col"].isna()]`: the rows where one column is missing.
> * `df.isna().any(axis=1)`: True for rows with at least one missing value.
> * `dropna()` and `fillna(value)` remove or fill them. **Deciding which, and why, is Week 4.**

In [32]:
# 🟢 How many missing, and what share, per column
missing = pd.DataFrame({"missing": df.isna().sum(), "share": df.isna().mean().round(3)})
missing[missing["missing"] > 0]

,missing,share
workclass,963,0.020
occupation,966,0.020
native_country,274,0.006


**What you should see:** missing values in `workclass`, `occupation` and `native_country`, each around 1 to 6% of rows. Every other column is complete.

In [33]:
# 🟢 Rows with any missing value, and the rows missing occupation
print(df.isna().any(axis=1).sum(), "rows have at least one missing value")
print(df[df["occupation"].isna()][["age", "workclass", "occupation"]].head(), "\n")

# 🟢 The trap: == never finds NaN
print((df["occupation"] == np.nan).sum(), "found with == np.nan;", df["occupation"].isna().sum(), "found with isna()")

1221 rows have at least one missing value
       age workclass occupation
32565   18       NaN        NaN
32567   29       NaN        NaN
32574   58       NaN        NaN
32583   72       NaN        NaN
32596   65       NaN        NaN 

0 found with == np.nan; 966 found with isna()


**What you should see:** a few thousand rows with at least one missing value; most rows missing `occupation` are also missing `workclass` (people outside the labour force, perhaps: a pattern worth investigating in Week 4); and `0 found with == np.nan` against the true count with `isna()`.

### Missing values in disguise

In the original CSV, missing values are written as `?`. If you read it **without** `na_values="?"`, they become ordinary text and `isna()` finds nothing. Here is the effect on a tiny example:

In [34]:
# 🟢 '?' as text, versus '?' as a missing value
from io import StringIO
text = "age,occupation\n39,Adm-clerical\n50,?\n38,Handlers-cleaners\n"
as_text = pd.read_csv(StringIO(text))
as_missing = pd.read_csv(StringIO(text), na_values="?")
print(as_text["occupation"].isna().sum(), "missing when '?' is text;",
      as_missing["occupation"].isna().sum(), "missing with na_values='?'")

0 missing when '?' is text; 1 missing with na_values='?'


> 🔵 **Try it.** Store in `share_missing_any` the share of rows (between 0 and 1) with at least one missing value. Then store in `complete` a DataFrame with only the rows that have **no** missing values.

In [35]:
share_missing_any = df.isna().any(axis=1).mean()
complete = df.dropna()
print(round(share_missing_any, 3), complete.shape)

0.025 (47621, 26)


In [36]:
check(np.isclose(share_missing_any, df.isna().any(axis=1).mean()), "Take .mean() of df.isna().any(axis=1).")
check(complete.isna().sum().sum() == 0 and len(complete) == len(df) - df.isna().any(axis=1).sum(), "dropna() keeps only complete rows.")

✅ Correct
✅ Correct


---
## 9. Sorting, counting and grouping

> **Method card**
> * `df.sort_values("col", ascending=False)`; `df.nlargest(n, "col")` for the top `n`.
> * `df.groupby("col")["other"].mean()`: one result per group. `.agg(name=("col", "func"), ...)` computes several at once.
> * `pd.crosstab(df.a, df.b, normalize="index")`: a table of counts, or of shares within each row.

In [37]:
# 🟢 The five longest working weeks
df.nlargest(5, "hours_per_week")[["age", "occupation", "hours_per_week", "income"]]

,age,occupation,hours_per_week,income
935,37,Craft-repair,99,>50K
1172,25,Farming-fishing,99,>50K
1887,55,Farming-fishing,99,>50K
3578,37,Prof-specialty,99,>50K
4086,50,?,99,<=50K


In [38]:
# 🟢 By education: number of people, mean hours, and share earning above 50K
by_edu = (df.groupby("education")
            .agg(people=("age", "size"),
                 mean_hours=("hours_per_week", "mean"),
                 share_high=("high_income", "mean"))
            .sort_values("share_high", ascending=False)
            .round(2))
by_edu.head(8)

,people,mean_hours,share_high
education,,,
Prof-school,834,47.58,0.74
Doctorate,594,46.58,0.73
Masters,2657,43.58,0.55
Bachelors,8025,42.48,0.41
Assoc-acdm,1601,40.81,0.26
Assoc-voc,2061,41.66,0.25
Some-college,10878,38.87,0.19
HS-grad,15784,40.64,0.16


**What you should see:** one row per education level, sorted by the share earning above 50K: doctorates and professional degrees first, with more than 70%, and the lowest levels at a few per cent. A clear gradient, as expected in income data.

In [39]:
# 🟢 Share above 50K by sex, within each sex
pd.crosstab(df["sex"], df["income"], normalize="index").round(3)

income,<=50K,>50K
sex,,
Female,0.891,0.109
Male,0.696,0.304


**What you should see:** about 11% of women and 30% of men in the data earn above 50K. That is a description of a 1994 sample, not a causal claim; the data also differ by occupation, hours and age, which you would examine together before drawing conclusions (Week 7, exploratory analysis).

> 🔵 **Try it.** Using `groupby`, compute the **median** `hours_per_week` for each `workclass`, sorted from highest to lowest. Store the Series in `hours_by_workclass`.

In [40]:
hours_by_workclass = df.groupby("workclass")["hours_per_week"].median().sort_values(ascending=False)
hours_by_workclass

,hours_per_week
workclass,
Self-emp-inc,50.0
State-gov,40.0
Federal-gov,40.0
Private,40.0
Local-gov,40.0
Without-pay,40.0
Self-emp-not-inc,40.0
?,36.0
Never-worked,35.0


In [41]:
check(hours_by_workclass.index[0] == df.groupby("workclass")["hours_per_week"].median().idxmax(), "groupby('workclass')['hours_per_week'].median(), then sort_values(ascending=False).")

✅ Correct


---
## 10. Types and memory, and saving

The text columns repeat a few values many times: `category` stores each value once (lecture, Part 2).

In [42]:
# 🟢 Text columns to category, and the memory saved
before = df.memory_usage(deep=True).sum()
text_cols = df.select_dtypes(exclude=["number", "bool"]).columns
typed = df.astype({c: "category" for c in text_cols})
after = typed.memory_usage(deep=True).sum()
print(f"{len(text_cols)} text columns -> category: {before / 1e6:.1f} MB -> {after / 1e6:.1f} MB")

12 text columns -> category: 38.7 MB -> 5.7 MB


In [43]:
# 🟢 Save: Parquet keeps the types; CSV would turn them back into text
typed.to_parquet("adult_typed.parquet")
print(pd.read_parquet("adult_typed.parquet").dtypes.value_counts())

int64       11
float64      2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
bool         1
category     1
Name: count, dtype: int64


---
## Summary: the pandas you used today

| Task | Code |
|---|---|
| Load through an API | `fetch_ucirepo(id=2)`; a CSV URL with `pd.read_csv(url, na_values=...)` |
| Look | `shape`, `head`, `info`, `describe`, `value_counts` |
| Columns | `df["a"]` (a Series), `df[["a", "b"]]` (a DataFrame) |
| Rows by label or position | `df.loc["a":"c", cols]` (end included), `df.iloc[0:2, 0]` (end excluded), `at`, `iat` |
| Filter | `df[(cond1) & (cond2)]`, `isin`, `between`, `str.contains(..., na=False)`, `query` |
| Arithmetic | `df["c"] = df["a"] - df["b"]`; Series and numbers broadcast; Series and Series align on labels |
| Functions | `map(dict)`, `apply(lambda x: ...)`, `apply(..., axis=1)` (slow), `assign(new=lambda d: ...)` |
| Missing | `isna().sum()`, `isna().any(axis=1)`, never `== np.nan` |
| Summarise | `sort_values`, `nlargest`, `groupby(...).agg(...)`, `crosstab` |
| Types | `astype("category")`, `memory_usage(deep=True)`, `to_parquet` |

**References.** Becker, B. and Kohavi, R. (1996), *Adult*, UCI Machine Learning Repository, https://doi.org/10.24432/C5XW20. McKinney, W. (2017), *Python for Data Analysis*, 2nd edition, chapter 5. pandas documentation, "10 minutes to pandas" and "Indexing and selecting data", pandas.pydata.org/docs.